# AfriVoices — KenLM v2 enrichi de texte externe (étage 2 vers 0.35)

**Session CPU** (pas d'unités GPU) — peut tourner en parallèle des tranches d'entraînement.

**Principe** : les KenLM actuels ne connaissent que les transcriptions d'entraînement.
On les enrichit de texte externe par langue → le LM corrige mieux, surtout les mots
qu'il n'a jamais vus dans les transcriptions.

**Sources par langue (honnêteté sur la disponibilité)** :
| Langue | Source externe | Poids test |
|---|---|---|
| swa | Wikipedia sw (riche) | 30 % |
| som | Wikipedia so | 9 % |
| kik | Wikipedia ki (petit) | 22 % |
| luo | MasakhaNEWS luo | 18 % |
| kln, mas | *aucune source publique fiable* → transcriptions seules | 21 % |

Les langues enrichissables couvrent **79 % du test**.

**Trois protections** : filtrage par l'alphabet exact du modèle (67 caractères — tout mot
hors-vocab est du poids mort), pondération domaine (transcriptions ×3 vs externe ×1 pour
ne pas noyer le registre oral), et sortie dans **lm_v2/** (les LM actuels restent intacts).

**Après** : re-tuning alpha/beta sur l'éval du modèle final (les optima bougent avec le LM).

## 1. Drive + alphabet du modèle (sans GPU, sans transformers)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json, os, re

BASE="/content/drive/MyDrive/afrivoices"
LM_V1=f"{BASE}/lm"          # corpus + LM actuels (transcriptions seules)
LM_V2=f"{BASE}/lm_v2"       # sortie enrichie
os.makedirs(LM_V2, exist_ok=True)

# alphabet exact du modèle depuis vocab.json (pas besoin de charger le modèle)
vocab=json.load(open(f"{BASE}/baobab-asr-v4/vocab.json", encoding="utf-8"))
ALPHABET=set(t for t in vocab.keys() if len(t)==1)   # caractères simples uniquement
ALPHABET.add(" ")
print("alphabet:", "".join(sorted(ALPHABET)))

def clean_ext(t):
    """Nettoyage agressif pour texte externe : minuscule + tout caractère hors
    alphabet devient un espace (coupe les mots pollués proprement)."""
    t=(t or "").lower()
    t="".join(ch if ch in ALPHABET else " " for ch in t)
    return re.sub(r"\s+"," ",t).strip()

def phrases(texte, min_mots=3, max_mots=30):
    """Découpe en phrases + filtre longueur."""
    out=[]
    for seg in re.split(r"[.!?;:\n]+", texte or ""):
        c=clean_ext(seg)
        n=len(c.split())
        if min_mots<=n<=max_mots: out.append(c)
    return out

# vérifier que les corpus v1 (transcriptions) existent
for l in ["swa","kik","luo","kln","mas","som"]:
    p=f"{LM_V1}/{l}.txt"
    print(l, "corpus v1:", os.path.exists(p), f"({os.path.getsize(p)//1024} Ko)" if os.path.exists(p) else "")

## 2. Collecte du texte externe par langue

Chaque source est dans un try/except : si une source est indisponible, on continue avec
ce qu'on a (et on le sait). Dédoublonnage à la fin.

In [ ]:
!pip install -q "datasets>=3.5,<4"
from datasets import load_dataset

externe={l:[] for l in ["swa","som","kik","luo","kln","mas"]}

# --- Wikipedia (dumps précompilés wikimedia/wikipedia) ---
WIKI={"swa":"20231101.sw", "som":"20231101.so", "kik":"20231101.ki"}
for lang,cfg in WIKI.items():
    try:
        ds=load_dataset("wikimedia/wikipedia", cfg, split="train")
        n0=len(externe[lang])
        for ex in ds:
            externe[lang]+=phrases(ex.get("text",""))
        print(f"{lang}: Wikipedia {cfg} -> +{len(externe[lang])-n0} phrases")
    except Exception as e:
        print(f"{lang}: Wikipedia indisponible ({str(e)[:70]})")

# --- MasakhaNEWS pour le luo ---
try:
    for split in ("train","validation","test"):
        ds=load_dataset("masakhane/masakhanews", "luo", split=split)
        for ex in ds:
            externe["luo"]+=phrases((ex.get("headline") or "")+". "+(ex.get("text") or ""))
    print(f"luo: MasakhaNEWS -> +{len(externe['luo'])} phrases")
except Exception as e:
    print(f"luo: MasakhaNEWS indisponible ({str(e)[:70]})")

# --- dédoublonnage ---
for l in externe:
    externe[l]=list(dict.fromkeys(externe[l]))
    print(f"{l}: {len(externe[l])} phrases externes (dédup)")

## 3. Fusion : transcriptions ×3 + externe ×1 → corpus v2

La répétition ×3 des transcriptions maintient la dominance du registre oral (domaine
cible). kln/mas : corpus v1 recopié tel quel (pas d'externe).

In [ ]:
for l in ["swa","kik","luo","kln","mas","som"]:
    v1=open(f"{LM_V1}/{l}.txt", encoding="utf-8").read().splitlines()
    v1=[t for t in v1 if t.strip()]
    corpus = v1*3 + externe[l]
    with open(f"{LM_V2}/{l}.txt","w",encoding="utf-8") as f:
        f.write("\n".join(corpus))
    print(f"{l}: {len(v1)} transcriptions x3 + {len(externe[l])} externes = {len(corpus)} lignes")

## 4. Compiler lmplz + entraîner les 5-grams v2

`--prune 0 0 1` élague les trigrams+ vus une seule fois : contient la taille des `.bin`
(important pour le swahili enrichi — pense à la contrainte RAM edge de 8 Go).

In [ ]:
!apt-get install -y build-essential cmake libboost-all-dev libeigen3-dev >/dev/null 2>&1
!git clone https://github.com/kpu/kenlm.git /content/kenlm 2>/dev/null
!cd /content/kenlm && mkdir -p build && cd build && cmake .. >/dev/null 2>&1 && make -j4 >/dev/null 2>&1
!ls /content/kenlm/build/bin/ | head -5

In [ ]:
import os
BIN="/content/kenlm/build/bin"
for lang in ["swa","kik","luo","kln","mas","som"]:
    txt, arpa, binf = f"{LM_V2}/{lang}.txt", f"{LM_V2}/{lang}.arpa", f"{LM_V2}/{lang}.bin"
    os.system(f"{BIN}/lmplz -o 5 --discount_fallback --prune 0 0 1 <{txt}> {arpa} 2>/tmp/{lang}.log")
    os.system(f"{BIN}/build_binary {arpa} {binf} 2>>/tmp/{lang}.log")
    ok=os.path.exists(binf) and os.path.getsize(binf)>0
    print(f"{lang}: {'OK' if ok else 'ECHEC -> voir /tmp/'+lang+'.log'}  bin={os.path.getsize(binf)//1e6 if ok else 0:.0f} Mo")
    if os.path.exists(arpa): os.remove(arpa)   # libérer le Drive (les .arpa sont énormes)

## 5. Test de cohérence rapide (perplexité relative)

Sanity check sans GPU : les LM v2 doivent scorer les transcriptions d'éval MIEUX (moins
négativement) que les LM v1, sinon l'enrichissement a dilué le domaine.

In [ ]:
!pip install -q https://github.com/kpu/kenlm/archive/master.zip
# ⚠️ redémarre le runtime après cette install, puis relance la cellule suivante

In [ ]:
import kenlm, os
BASE="/content/drive/MyDrive/afrivoices"
for lang in ["swa","kik","luo","som"]:
    # échantillon de transcriptions (fin de fichier v1 = jamais vues en tête de corpus)
    lines=open(f"{BASE}/lm/{lang}.txt",encoding="utf-8").read().splitlines()[-500:]
    m1=kenlm.Model(f"{BASE}/lm/{lang}.bin")
    m2=kenlm.Model(f"{BASE}/lm_v2/{lang}.bin")
    s1=sum(m1.score(l) for l in lines)/len(lines)
    s2=sum(m2.score(l) for l in lines)/len(lines)
    print(f"{lang}: v1 {s1:.1f} | v2 {s2:.1f} -> {'v2 mieux' if s2>s1 else 'v1 mieux (dilution?)'}")

## 6. Utilisation avec le modèle final

Quand les tranches v5 seront finies : dans le bloc de construction des décodeurs, change
simplement `LM = ".../lm"` en `LM = ".../lm_v2"`, puis **refais le tuning alpha/beta**
sur l'éval (grille autour de 0.5-0.9 / 0.5-1.5 — les optima bougent avec un LM plus gros).

Compare sur l'éval interne : modèle final + LM v1 vs + LM v2. Ne soumets que le meilleur.

⚠️ Vérifie la taille du swa.bin v2 pour le rapport edge : modèle + plus gros LM doit
rester ≤ 8 Go (large marge attendue, mais à re-mesurer pour le livrable).